![UTN Facultad Regional Mendoza](https://raw.githubusercontent.com/javovelez/Modelos-de-Lenguaje/main/img/logo_utn_frm.png)

# Laboratorio n° 4. Destilar un razonador

**Asignatura:** Modelos de Lenguaje
**Bloque:** 4. LLMs

---

## Introducción

`Qwen3-0.6B` es el modelo `Qwen3-0.6B-Base` después del post-entrenamiento de Qwen. Ese post-entrenamiento lo ajustó para escribir un pensamiento entre `<think>` y `</think>` antes de la respuesta, como muestra la sección 1.1 de la Clase 2. El modelo base no tiene ese post-entrenamiento. Por eso el modelo base completa texto y no escribe un pensamiento antes de responder.

La **destilación**, presentada en la sección 3.4 de la Clase 2, entrena un modelo pequeño con los textos que escribe un modelo más grande. El modelo que escribe los textos se llama **maestro**. El modelo que se entrena se llama **alumno**. En este laboratorio, «alumno» nombra siempre a ese modelo y nunca a quien resuelve el laboratorio. En este laboratorio, el maestro es `Qwen3-4B` con el razonamiento encendido y el alumno es `Qwen3-0.6B-Base`.

**GSM8K** es un conjunto de problemas de matemática de nivel primario. Cada problema es un enunciado de texto cuya solución lleva entre dos y ocho pasos de aritmética básica. La respuesta es siempre un número. El conjunto tiene 7.473 problemas de entrenamiento y 1.319 de prueba.

Una **traza** es el pensamiento y la respuesta que el maestro escribió para un problema. El laboratorio entrega 491 trazas ya generadas, cada una sobre un problema distinto del conjunto de entrenamiento de GSM8K, traducido al español. Generarlas llevó unas 4,5 horas de GPU. El maestro escribe el pensamiento en inglés y la respuesta en español.

Cada traza pasó un filtro como el del muestreo por rechazo de la sección 3.3 de la Clase 2. El filtro conserva solo las trazas con la respuesta correcta y el pensamiento cerrado. Además, cada traza ocupa menos de 1.024 tokens. En el archivo, las trazas miden entre 276 y 972 tokens, con una mediana de 690.

**MGSM** es una traducción de GSM8K a diez idiomas, hecha por traductores humanos sobre 250 problemas del conjunto de prueba. **MGSM-es** es su versión en español. El laboratorio mide cada modelo sobre esos 250 problemas, con el verificador de la sección 1.2 de la Clase 2.

Las trazas salen del conjunto de entrenamiento de GSM8K, así que ningún problema de la evaluación aparece en las trazas.

La medición de cada modelo se compara con dos medidas fijas.

- **El piso** es el modelo base sin entrenar, con el problema escrito como texto a completar y sin plantilla de chat. La sección 3.4 de la Clase 2 llama a esta medida el piso honesto. Si el modelo base se midiera con una plantilla que no conoce, cualquier entrenamiento parecería mejor de lo que es.
- **`Qwen3-0.6B`** es el mismo modelo con el post-entrenamiento completo de Qwen, preguntado con la plantilla de chat y el razonamiento encendido.

Al completar este laboratorio vas a poder:

- Construir el texto que el modelo debe aprender a producir para una traza, y verificar en qué posiciones cuenta la pérdida.
- Entrenar un modelo alumno con LoRA y medirlo contra el piso, problema por problema.
- Diagnosticar una falla de entrenamiento a partir de las respuestas generadas y de los pesos del modelo.
- Explicar qué matrices del modelo ajusta LoRA.
- Interpretar una diferencia de accuracy con la escala del ruido entre corridas.

---

## Instrucciones generales

- Completá el código en las celdas marcadas con `# Tu código aquí`.
- Respondé las preguntas de análisis en las celdas de texto (tipo Markdown).
- **El laboratorio necesita GPU.** En Colab, *Entorno de ejecución > Cambiar tipo de entorno > T4 GPU*. El laboratorio entrena tres modelos alumno y evalúa cada uno sobre 250 problemas. En una T4, cada entrenamiento lleva del orden de 17 minutos y cada evaluación del orden de 20. El total es del orden de dos horas de GPU. Las evaluaciones del piso y de `Qwen3-0.6B` vienen medidas por la cátedra. Antes de empezar, verificá que la primera celda de preparación imprima `cuda` como dispositivo.
- **El trabajo se puede repartir en varias sesiones.** La primera celda de preparación pide acceso a Google Drive y usa la carpeta `modelos-lenguaje-u4`. Cada entrenamiento guarda en esa carpeta los pesos que entrenó. Cada evaluación guarda sus respuestas cada 50 problemas. Si la sesión se interrumpe, al volver a ejecutar las celdas los entrenamientos terminados se cargan y las evaluaciones siguen desde el último tramo guardado. Si no autorizás el acceso, la carpeta se crea en el disco de la sesión y se pierde al cerrarla.
- Este laboratorio va después de las dos clases de la Unidad 4. Da por vistos la plantilla de chat (sección 3.1 de la Clase 1), LoRA y la pérdida calculada solo sobre la respuesta (sección 3.3 de la Clase 1), el razonamiento entre `<think>` y `</think>` (sección 1.1 de la Clase 2), el verificador (sección 1.2 de la Clase 2) y el presupuesto de tokens (sección 2.2 de la Clase 2). Lo que el laboratorio usa y la clase no presentó, el laboratorio lo explica antes de pedirlo.
- Todas las evaluaciones usan decodificación voraz, para que la comparación entre modelos no dependa del muestreo. Aun así, una diferencia numérica mínima entre dos placas cambia el token elegido en alguna posición. Desde ese token cambia toda la cadena de razonamiento. Por eso tus cifras van a diferir de las de otro grupo. Lo que se corrige es la dirección de cada resultado y el argumento con el que lo explicás.

## IMPORTANTE. Qué celdas podés modificar

Este laboratorio es un **entregable**. Solo debés completar las celdas de actividad, que son las que aparecen con el comentario `# Tu código aquí` o el texto `*(Escribí tu respuesta acá)*`. Las demás celdas, es decir, los enunciados, las explicaciones, los ejemplos provistos y el encabezado, **no se tocan**.

La corrección se hace celda por celda. Cada respuesta se busca en la celda donde el enunciado la pide. Si escribís en otro lado, o si movés, renombrás o borrás celdas del enunciado, esa parte de tu entrega queda sin poder corregirse.

Si querés probar algo por separado, hacelo en la misma celda de actividad o en una celda nueva que agregues, y borrala antes de entregar.

---
## Preparación

Las dos celdas que siguen vienen resueltas. No hay nada que completar en ellas, pero hay que ejecutarlas en orden antes de empezar. Cada una lleva antes una presentación con los nombres que deja definidos. Los ejercicios usan esos nombres.

La primera celda instala las bibliotecas que falten y las importa. También descarga `u4c2_comun.py`, un archivo con las funciones de la Clase 2 que este laboratorio reutiliza, y lo importa con el nombre `c2`. Por último, prepara la carpeta de resultados. La segunda celda carga las trazas, los problemas de MGSM-es y el tokenizador del modelo base.

El laboratorio tiene tres celdas de preparación más, ubicadas junto a los ejercicios que las necesitan. La primera, después del Ejercicio 1, define el entrenamiento y la evaluación. La segunda, antes del Ejercicio 2, carga la evaluación del piso. La tercera, antes del Ejercicio 3, carga la evaluación de `Qwen3-0.6B`.

**El tipo de los pesos.** Una T4 no tiene `bfloat16` en el *hardware*. En una T4, la cátedra midió que evaluar con los pesos en `float16` es 2,5 veces más rápido que en `bfloat16`, y que entrenar es 3,4 veces más rápido. Por eso la celda elige el tipo según la placa. Con las placas que sí tienen `bfloat16`, el tipo es `bfloat16`, como en las dos clases.

| Nombre | Qué es |
|---|---|
| `DISPOSITIVO` | `"cuda"` si hay GPU; en otro caso, `"mps"` o `"cpu"` |
| `TIPO` | el tipo de los pesos al evaluar, `torch.float16` en una T4 y `torch.bfloat16` en una placa que lo tenga |
| `c2` | el archivo `u4c2_comun.py`, que reúne las funciones de la Clase 2 que usa el laboratorio, como el verificador de respuestas, `evaluar` y `comparar`. Se las llama con el prefijo `c2.`, por ejemplo `c2.evaluar` |
| `BASE`, `POST` | los nombres en el Hub de `Qwen3-0.6B-Base` y de `Qwen3-0.6B` |
| `CARPETA` | la carpeta donde se guardan los entrenamientos y las evaluaciones |

In [ ]:
# ─── Preparación. Bibliotecas, funciones de la Clase 2 y carpeta de resultados ─
import os, sys, math, time, json, contextlib, importlib, subprocess, urllib.request, warnings
from importlib.metadata import version, PackageNotFoundError


def instalar(*argumentos):
    """Instala con el pip del intérprete del notebook. Si pip falla, corta y muestra su error."""
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *argumentos],
                       capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(f"pip install {' '.join(argumentos)} falló:\n{r.stderr[-3000:]}")
    importlib.invalidate_caches()                     # para que el import vea el paquete recién instalado


def version_de(paquete):
    """La versión instalada como tupla de enteros, o (0,) si el paquete no está."""
    try:
        return tuple(int(x) for x in version(paquete).split(".")[:2] if x.isdigit())
    except PackageNotFoundError:
        return (0,)


if version_de("transformers") < (5,):                # transformers 5 o posterior, como las dos clases
    instalar("-U", "transformers>=5")
if version_de("peft") < (0, 21):                     # merge_and_unload con trainable_token_indices pide 0.21
    instalar("-U", "peft>=0.21")
if version_de("datasets") < (3,):                    # las funciones de la Clase 2 leen MGSM-es con datasets
    instalar("-U", "datasets>=3")
if (0,) < version_de("torchao") < (0, 16):          # Colab trae un torchao con el que peft no importa
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], check=False)

os.environ.setdefault("HF_HUB_VERBOSITY", "error")
warnings.filterwarnings("ignore", message="IProgress not found")

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import transformers
import peft
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, PeftModel, get_peft_model

transformers.logging.set_verbosity_error()
transformers.utils.logging.disable_progress_bar()
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 250)

# El repositorio de la materia da una URL estable. El Drive de la cátedra queda de
# respaldo, porque Drive corta las descargas públicas por cuota.
RELEASES = "https://github.com/javovelez/Modelos-de-Lenguaje/releases/download"
DATOS = "https://github.com/javovelez/Modelos-de-Lenguaje/raw/main/datos"
FUENTES = {                                     # archivo: (URL, ID de Drive)
    "cache_modelos.py": (f"{RELEASES}/utils-v1/cache_modelos.py", "1k5wxbSMS-_3dC8FN_F-2VN6dp8tGFCpz"),
    "u4c2_comun.py":    (f"{RELEASES}/utils-v1/u4c2_comun.py", "17q-7iVU5n7BzYtfcgG4GrNbVkcEJ-fNm"),
    "trazas_tp.jsonl":  (f"{DATOS}/trazas_tp.jsonl", "1U1p2hVlD4HZGe55Xh75NP2w5OcEmR6EV"),
}


def descargar(nombre):
    """Descarga un archivo de FUENTES, salvo que ya exista. Si la URL no responde, usa Drive."""
    if os.path.exists(nombre):
        return
    url, id_drive = FUENTES[nombre]
    try:
        urllib.request.urlretrieve(url, nombre)
    except Exception as falla:
        print(f"el repositorio de la materia no respondió ({falla}); se usa el Drive.")
        try:
            import gdown
        except ImportError:
            instalar("gdown")
            import gdown
        gdown.download(id=id_drive, output=nombre, quiet=True)


descargar("cache_modelos.py")
descargar("u4c2_comun.py")
from cache_modelos import preparar_carpeta
import u4c2_comun as c2

if torch.cuda.is_available():
    DISPOSITIVO = "cuda"
elif torch.backends.mps.is_available():
    DISPOSITIVO = "mps"
else:
    DISPOSITIVO = "cpu"
c2.DISPOSITIVO = torch.device(DISPOSITIVO)          # las funciones de la Clase 2 solo distinguen entre cuda y cpu

# torch.cuda.is_bf16_supported() no sirve para elegir: en una T4 devuelve True, porque cuenta la emulación.
if DISPOSITIVO == "cuda":
    TIPO = torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16
else:
    TIPO = torch.float32

BASE, POST = "Qwen/Qwen3-0.6B-Base", "Qwen/Qwen3-0.6B"
CARPETA = preparar_carpeta("modelos-lenguaje-u4")
if DISPOSITIVO == "cpu":
    print("Sin GPU. Cambiá el tipo de entorno a T4 antes de seguir.")

pd.Series({"transformers": transformers.__version__, "peft": peft.__version__, "torch": torch.__version__,
           "dispositivo": DISPOSITIVO, "tipo": str(TIPO).replace("torch.", "")}).to_frame("versión")

### Las trazas y MGSM-es

Cada traza es un diccionario. El laboratorio usa cuatro de sus claves.

- `"problema"` es el texto del problema, en español.
- `"respuesta_real"` es la respuesta correcta, un número.
- `"pensamiento"` es el texto que el maestro escribió entre `<think>` y `</think>`, sin esas dos marcas.
- `"respuesta_modelo"` es el texto que el maestro escribió después de `</think>`.

Las trazas están en un orden al azar que se decidió una sola vez, y ese orden quedó guardado en el archivo. No cambia de una corrida a otra. Por eso `trazas[:100]`, las primeras 100 trazas, son siempre las mismas, para todos los grupos y en cualquier sesión.

Esto importa en dos lugares. Los ejercicios 2 y 4 entrenan con esas 100 trazas, así que la diferencia entre ellos se debe al entrenamiento de las filas y no a que hayan tocado trazas distintas. El Ejercicio 5 compara las 100 trazas con las 491, y las 100 son un subconjunto de las 491.

Los problemas de MGSM-es se cargan con `c2.cargar_mgsm`, la función de la Clase 2. Cada problema es un diccionario con la pregunta en la clave `"pregunta"` y la respuesta correcta en la clave `"respuesta"`.

Los cuatro **tokens especiales** de la plantilla de chat de Qwen3 son `<think>`, `</think>`, `<|im_start|>` y `<|im_end|>`. La sección 3.1 de la Clase 1 presenta `<|im_start|>` y `<|im_end|>`, que abren y cierran cada turno de la conversación. Cada uno de los cuatro es un solo token del vocabulario.

| Nombre | Qué es |
|---|---|
| `trazas` | la lista de las 491 trazas |
| `prueba` | la lista de los 250 problemas de MGSM-es |
| `tok_base` | el tokenizador de `Qwen3-0.6B-Base`, que es el mismo de `Qwen3-0.6B` |
| `ESPECIALES` | la lista de los textos de los cuatro tokens especiales |
| `IDS_ESPECIALES` | la lista de los índices de esos cuatro tokens en el vocabulario, en el mismo orden |
| `prompt_chat(pregunta)` | la función que recibe el texto de un problema, `pregunta`, y devuelve el prompt como un texto. El prompt tiene el problema como turno del usuario, con la plantilla de chat y el razonamiento encendido, y termina en `<\|im_start\|>assistant\n`, donde empieza la respuesta del modelo |
| `prompt_texto(pregunta)` | la función que recibe el texto de un problema, `pregunta`, y devuelve el prompt como un texto a completar, sin plantilla de chat. Es el prompt del piso |

In [ ]:
# ─── Preparación. Las trazas, MGSM-es y el tokenizador ─────────────────────
descargar("trazas_tp.jsonl")
with open("trazas_tp.jsonl", encoding="utf-8") as archivo:
    trazas = [json.loads(linea) for linea in archivo]
prueba, _ = c2.cargar_mgsm()                          # los 250 problemas de prueba de MGSM-es

tok_base = AutoTokenizer.from_pretrained(BASE)
ESPECIALES = ["<think>", "</think>", "<|im_start|>", "<|im_end|>"]
IDS_ESPECIALES = tok_base.convert_tokens_to_ids(ESPECIALES)


def prompt_chat(pregunta):
    """El problema como turno del usuario, con la plantilla de chat y el razonamiento encendido."""
    return c2.prompt_de_chat(tok_base, pregunta, pensar=True)


def prompt_texto(pregunta):
    """El problema como texto a completar, sin plantilla de chat. Es el prompt del piso."""
    return f"Pregunta: {pregunta}\nRespuesta paso a paso:"


print(f"{len(trazas)} trazas y {len(prueba)} problemas de MGSM-es")
print("tokens especiales:", dict(zip(ESPECIALES, IDS_ESPECIALES)))
print()
print("problema:          ", trazas[0]["problema"])
print("respuesta correcta:", trazas[0]["respuesta_real"])
print("pensamiento:       ", trazas[0]["pensamiento"][:400], "[…]")
print("respuesta:         ", trazas[0]["respuesta_modelo"][-300:])

---
### Ejercicio 1. El objetivo de entrenamiento

**Objetivo.** Escribir la función que arma el texto que el modelo tiene que aprender a producir para una traza, y verificar sobre qué posiciones se calcula la pérdida.

El entrenamiento de la sección 3.3 de la Clase 1 concatena un prompt y una respuesta. La pérdida de ese entrenamiento se calcula solo sobre las posiciones de la respuesta. En este laboratorio, el prompt es el texto que devuelve `prompt_chat`, que contiene el problema con la plantilla de chat y el razonamiento encendido. El texto que el modelo alumno tiene que escribir después del prompt se llama **objetivo de entrenamiento**.

El objetivo tiene el formato de las respuestas de `Qwen3-0.6B` en la sección 1.1 de la Clase 2. Termina con `<|im_end|>`, que cierra el turno del asistente.

**Enunciado:**

1. Escribí la función `objetivo(traza)`. La función recibe una traza, que es un diccionario de la lista `trazas`, y devuelve el texto `"<think>\n{pensamiento}\n</think>\n\n{respuesta}<|im_end|>"`. En ese texto, `{pensamiento}` es el valor de la clave `"pensamiento"` de la traza y `{respuesta}` es el valor de su clave `"respuesta_modelo"`. La función de entrenamiento de la preparación siguiente construye con tu función el objetivo de cada traza.
2. Construí un `DataFrame` llamado `etiquetas` para la primera traza, siguiendo estos pasos en orden. Es la misma tabla de la sección 3.3 de la Clase 1.
   - Codificá el prompt de la traza con `tok_base.encode`. El resultado es la lista de los índices de sus tokens.
   - Codificá el objetivo de la traza de la misma manera.
   - Armá la lista de las etiquetas. Tiene un −100 por cada token del prompt, seguido de los índices de los tokens del objetivo. El valor −100 indica que la pérdida no cuenta esa posición.
   - Armá la lista de los tokens del texto completo, que son los del prompt seguidos de los del objetivo. Construí el `DataFrame` con una fila por token y dos columnas.
      - `"token"` es el texto del token, que se obtiene con `tok_base.decode`.
      - `"etiqueta"` es el valor de la lista de etiquetas en esa posición.
3. Imprimí la cantidad de posiciones del prompt y la del objetivo. Mostrá dos tramos de `etiquetas`. El primero tiene las cuatro últimas posiciones del prompt y las cuatro primeras del objetivo. El segundo tiene las cuatro últimas posiciones del objetivo.

Verificá que `<think>`, en el primer tramo, y `<|im_end|>`, en el segundo, aparezcan como un token cada uno. Si alguno aparece partido en varios tokens, el texto que escribiste en `objetivo` no es el de ese token.

In [ ]:
# Tu código aquí

**Pregunta de análisis:**

La pérdida cuenta también la última posición del objetivo, la de `<|im_end|>`. ¿Qué ajusta el entrenamiento en esa posición, y qué pasaría en la evaluación si el objetivo terminara en la respuesta, sin `<|im_end|>`?

*(Escribí tu respuesta acá)*

---
### Preparación. El entrenamiento y la evaluación

La celda que sigue define las funciones que usan los ejercicios 2, 4 y 5. Las dos principales son `entrenar_alumno`, que entrena un alumno, y `evaluar_alumno`, que lo evalúa.

`entrenar_alumno` aplica la receta de la sección 3.3 de la Clase 1. LoRA se aplica a las siete proyecciones de cada bloque, con r = 16 y alfa = 32. El optimizador es AdamW, con un `lr` de `1e-4`. Cada paso de optimización acumula los gradientes de 4 lotes de 2 trazas, así que el lote efectivo es de 8 trazas. Los tres alumnos del laboratorio entrenan 220 pasos, cualquiera sea la cantidad de trazas. Cuando el entrenamiento termina de recorrer las trazas, las vuelve a recorrer en otro orden al azar. La pérdida se calcula solo sobre las posiciones del objetivo, construido con tu función `objetivo` del Ejercicio 1.

En una T4, los pesos en `float16` necesitan un cuidado más para entrenar. El rango de `float16` es mucho menor que el de `float32`, así que un gradiente muy cercano a cero se redondea a cero. Por eso, en una T4, `entrenar_alumno` carga el modelo en `float32` y calcula en `float16` solo las operaciones del modelo, que es la precisión mixta. El escalador de gradientes `GradScaler` multiplica la pérdida por un factor grande antes de calcular los gradientes y los divide por el mismo factor antes del paso del optimizador.

`entrenar_alumno` guarda en `CARPETA`, en una carpeta con el nombre del alumno, los pesos entrenados y un archivo con la configuración, la curva de la pérdida y el tiempo. Si esos archivos existen y la configuración coincide, la función los carga y no entrena. La configuración incluye el texto del prompt y del objetivo de la primera traza. Si corregís tu función `objetivo` después de entrenar, la configuración cambia y la función vuelve a entrenar.

`evaluar_alumno` carga el modelo base, le suma los pesos entrenados y lo evalúa con `evaluar_o_cargar`. `evaluar_o_cargar` evalúa los 250 problemas con `c2.evaluar`, de a 50, con decodificación voraz y un presupuesto de 1.024 tokens. El prompt es el que devuelve `prompt_chat`. Después de cada tramo de 50 problemas, la función agrega las respuestas a un archivo en `CARPETA`. Si la sesión se interrumpe, la función sigue desde el último tramo guardado.

| Nombre | Qué es |
|---|---|
| `PASOS`, `PRESUPUESTO` | 220 pasos de optimización y 1.024 tokens de presupuesto |
| `cargar(nombre, tipo=TIPO, dispositivo=DISPOSITIVO)` | la función que devuelve el tokenizador y el modelo `nombre` del Hub, con los pesos de tipo `tipo` en `dispositivo` |
| `entrenar_alumno(nombre, trazas_usadas, tokens_especiales=None)` | la función que entrena un alumno con las trazas de la lista `trazas_usadas`, o carga el entrenamiento guardado. El argumento `tokens_especiales` se presenta en el Ejercicio 4. La función devuelve un diccionario con las claves `"nombre"`, `"entrenables"`, `"minutos"` e `"historia"`, que es la curva de la pérdida |
| `evaluar_alumno(info)` | la función que evalúa el alumno descrito por `info`, el diccionario que devolvió `entrenar_alumno`. La función devuelve el resultado de `evaluar_o_cargar` |
| `evaluar_o_cargar(nombre, cargar_modelo, armar)` | la función que evalúa un modelo sobre los 250 problemas, o carga la evaluación guardada. `cargar_modelo` es una función sin argumentos que devuelve el tokenizador y el modelo. `armar` es la función que construye el prompt de cada problema. El resultado es un diccionario con las claves `"accuracy"`, `"tokens_por_respuesta"`, `"minutos"` y `"detalle"`. `"detalle"` tiene un diccionario por problema, como el de `c2.evaluar` |
| `resumen(m, info=None)` | la función que recibe un resultado `m` de `evaluar_o_cargar` y, si es un alumno, su `info`. La función devuelve una `Series` con la fila de `m` en la tabla `comparacion`, que crea la preparación siguiente |

La fila de `resumen` tiene ocho columnas, que son la accuracy, los puntos sobre el piso, los problemas que el modelo resuelve y el piso no, los problemas que el piso resuelve y el modelo no, los tokens por respuesta, los parámetros entrenables en miles y los minutos de entrenamiento y de evaluación. Los dos conteos de problemas salen de `c2.comparar`, la función de la Clase 2.

In [ ]:
# ─── Preparación. El entrenamiento con LoRA y la evaluación por tramos ──────
PASOS, LOTE, ACUMULAR, LR = 220, 2, 4, 1e-4          # lote efectivo de 8 trazas
LARGO_MAX = 1200                                      # la traza más larga, con su prompt, ocupa 1.024 tokens
PROYECCIONES = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
PRESUPUESTO, LOTE_EVAL, TRAMO_EVAL = 1024, 16, 50
# En float16 se entrena con el modelo en float32 y precisión mixta. Cargado en float16, GradScaler
# rechazaría los gradientes de las filas de los tokens especiales del Ejercicio 4.
CARGA_ENTRENAR = torch.float32 if TIPO == torch.float16 else TIPO


def cargar(nombre, tipo=TIPO, dispositivo=DISPOSITIVO):
    """Tokenizador con relleno a la izquierda, para generar por lotes, y modelo en modo evaluación."""
    tok = AutoTokenizer.from_pretrained(nombre)
    tok.padding_side = "left"
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    modelo = AutoModelForCausalLM.from_pretrained(nombre, dtype=tipo)
    return tok, modelo.to(dispositivo).eval()


def entrenar_alumno(nombre, trazas_usadas, tokens_especiales=None, semilla=0):
    """Entrena con LoRA (y las filas de tokens_especiales) o carga el entrenamiento guardado."""
    ejemplo = trazas_usadas[0]
    config = {"modelo": BASE, "trazas": len(trazas_usadas), "pasos": PASOS, "lote": LOTE, "acumular": ACUMULAR,
              "lr": LR, "r": 16, "alfa": 32, "tokens_especiales": tokens_especiales, "semilla": semilla,
              "carga": str(CARGA_ENTRENAR), "ejemplo": prompt_chat(ejemplo["problema"]) + objetivo(ejemplo)}
    ruta = os.path.join(CARPETA, nombre)
    ruta_info = os.path.join(ruta, "entrenamiento.json")
    if os.path.exists(ruta_info):
        with open(ruta_info, encoding="utf-8") as archivo:
            info = json.load(archivo)
        if info["config"] == config:
            print(f"[{nombre}] cargado de {ruta} (entrenado el {info['fecha']})")
            return info
        print(f"[{nombre}] cambió la configuración o el objetivo: se vuelve a entrenar")

    torch.manual_seed(semilla)
    tok, modelo = cargar(BASE, CARGA_ENTRENAR)
    modelo.train()
    modelo.gradient_checkpointing_enable()
    modelo.config.use_cache = False
    modelo = get_peft_model(modelo, LoraConfig(
        r=16, lora_alpha=32, lora_dropout=0.05, task_type="CAUSAL_LM", target_modules=PROYECCIONES,
        trainable_token_indices=tokens_especiales))
    entrenables = sum(p.numel() for p in modelo.parameters() if p.requires_grad)

    ids_p = [tok.encode(prompt_chat(t["problema"]), add_special_tokens=False) for t in trazas_usadas]
    ids_o = [tok.encode(objetivo(t), add_special_tokens=False)[:LARGO_MAX - len(p)]
             for t, p in zip(trazas_usadas, ids_p)]
    parametros = [p for p in modelo.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(parametros, lr=LR, weight_decay=0.0)
    calentamiento = max(1, PASOS // 10)
    programa = torch.optim.lr_scheduler.LambdaLR(   # multiplicador del lr: de 0 a 1, y de 1 a 0
        opt, lambda p: min((p + 1) / calentamiento, max(0.0, (PASOS - p) / (PASOS - calentamiento))))
    escalar = TIPO == torch.float16
    escalador = torch.amp.GradScaler("cuda" if DISPOSITIVO == "cuda" else "cpu", enabled=escalar)
    con_autocast = DISPOSITIVO == "cuda" and TIPO != torch.float32

    # Las trazas en el orden en que se usan: tantas permutaciones como hagan falta para los PASOS.
    rng = np.random.default_rng(semilla)
    necesarias = PASOS * LOTE * ACUMULAR
    orden = np.concatenate([rng.permutation(len(trazas_usadas))
                            for _ in range(math.ceil(necesarias / len(trazas_usadas)))])[:necesarias]
    historia, acumulada, t0 = [], 0.0, time.time()
    print(f"[{nombre}] entrenando {PASOS} pasos en {DISPOSITIVO}", flush=True)
    for paso in range(PASOS):
        for micro in range(ACUMULAR):
            inicio = (paso * ACUMULAR + micro) * LOTE
            filas = [(ids_p[k], ids_o[k]) for k in orden[inicio:inicio + LOTE]]
            L = max(len(p) + len(o) for p, o in filas)
            x = torch.full((len(filas), L), tok.pad_token_id)
            y = torch.full((len(filas), L), -100)
            for f, (p, o) in enumerate(filas):          # relleno a la derecha; la pérdida, solo en el objetivo
                x[f, :len(p) + len(o)] = torch.tensor(p + o)
                y[f, len(p):len(p) + len(o)] = torch.tensor(o)
            x, y = x.to(DISPOSITIVO), y.to(DISPOSITIVO)
            with torch.autocast("cuda", dtype=TIPO) if con_autocast else contextlib.nullcontext():
                logits = modelo(input_ids=x, attention_mask=(x != tok.pad_token_id).long()).logits
            cuenta = y[:, 1:] != -100                   # se eligen las posiciones antes de pasar a float32
            perdida = F.cross_entropy(logits[:, :-1][cuenta].float(), y[:, 1:][cuenta])
            escalador.scale(perdida / ACUMULAR).backward()
            acumulada += perdida.item() / ACUMULAR
            del x, y, logits
        escalador.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(parametros, 1.0)
        escalador.step(opt)
        escalador.update()
        programa.step()
        opt.zero_grad(set_to_none=True)
        if DISPOSITIVO == "cuda":
            torch.cuda.empty_cache()                    # los lotes de largos distintos fragmentan la memoria
        if (paso + 1) % 20 == 0:
            historia.append({"paso": paso + 1, "pérdida": round(acumulada / 20, 4)})
            print(f"   paso {paso + 1:4d}/{PASOS}  pérdida {acumulada / 20:.3f}", flush=True)
            acumulada = 0.0

    info = {"nombre": nombre, "config": config, "entrenables": entrenables, "historia": historia,
            "minutos": round((time.time() - t0) / 60, 1), "fecha": time.strftime("%Y-%m-%d %H:%M")}
    modelo.save_pretrained(ruta)
    with open(ruta_info, "w", encoding="utf-8") as archivo:
        json.dump(info, archivo, ensure_ascii=False)
    print(f"[{nombre}] {info['minutos']} minutos, guardado en {ruta}")
    del modelo, opt
    c2.liberar()
    return info


def cargar_alumno(nombre):
    """El modelo base con los pesos entrenados del alumno sumados, en el tipo de evaluación."""
    tok, modelo = cargar(BASE)
    modelo = PeftModel.from_pretrained(modelo, os.path.join(CARPETA, nombre)).merge_and_unload()
    modelo.config.use_cache = True
    return tok, modelo.eval()


def evaluar_o_cargar(nombre, cargar_modelo, armar, **config):
    """Evalúa los 250 problemas de a TRAMO_EVAL y guarda cada tramo; si la evaluación está guardada, la carga."""
    config = {"problemas": len(prueba), "presupuesto": PRESUPUESTO, "tipo": str(TIPO),
              "ejemplo": armar(prueba[0]["pregunta"]), **config}
    ruta = os.path.join(CARPETA, f"{nombre}.jsonl")
    detalle = []
    if os.path.exists(ruta):
        with open(ruta, encoding="utf-8") as archivo:
            lineas = [json.loads(linea) for linea in archivo if linea.strip()]
        if lineas and lineas[0] == {"config": config}:
            detalle = lineas[1:]
        else:
            print(f"[{nombre}] cambió la configuración: se vuelve a evaluar")
    if len(detalle) < len(prueba):
        if detalle:
            print(f"[{nombre}] se retoma la evaluación desde el problema {len(detalle)}")
        tok, modelo = cargar_modelo()
        with open(ruta, "a" if detalle else "w", encoding="utf-8") as archivo:
            if not detalle:
                archivo.write(json.dumps({"config": config}, ensure_ascii=False) + "\n")
            for i0 in range(len(detalle), len(prueba), TRAMO_EVAL):
                t0 = time.time()
                m = c2.evaluar(modelo, tok, prueba[i0:i0 + TRAMO_EVAL], armar, PRESUPUESTO, lote=LOTE_EVAL)
                segundos = (time.time() - t0) / len(m["detalle"])
                for d in m["detalle"]:
                    d["segundos"] = segundos
                    archivo.write(json.dumps(d, ensure_ascii=False) + "\n")
                    detalle.append(d)
                archivo.flush()
                print(f"[{nombre}] {len(detalle)}/{len(prueba)} problemas, "
                      f"{sum(d['ok'] for d in detalle)} resueltos", flush=True)
        del modelo
        c2.liberar()
    else:
        print(f"[{nombre}] cargado de {ruta}")
    return resultado(nombre, detalle)


def resultado(nombre, detalle):
    """El resultado de una evaluación a partir de su detalle por problema."""
    return {"nombre": nombre, "detalle": detalle,
            "accuracy": round(100 * sum(d["ok"] for d in detalle) / len(detalle), 1),
            "tokens_por_respuesta": round(float(np.mean([d["tokens"][0] for d in detalle])), 1),
            "minutos": round(sum(d["segundos"] for d in detalle) / 60, 1)}


def evaluar_alumno(info):
    """Evalúa el alumno que entrenó entrenar_alumno, con la plantilla de chat."""
    return evaluar_o_cargar(info["nombre"], lambda: cargar_alumno(info["nombre"]), prompt_chat,
                            modelo=info["nombre"], entrenado=info["fecha"])


def resumen(m, info=None):
    """La fila de la tabla comparacion para un resultado de evaluar_o_cargar."""
    contra_piso = c2.comparar(m, piso)
    return pd.Series({"accuracy %": m["accuracy"],
                      "sobre el piso": round(m["accuracy"] - piso["accuracy"], 1),
                      "resuelve y el piso no": contra_piso["gana A"],
                      "el piso resuelve y no": contra_piso["gana B"],
                      "tokens por respuesta": m["tokens_por_respuesta"],
                      "entrenables (miles)": round(info["entrenables"] / 1e3, 1) if info else None,
                      "minutos de entrenamiento": info["minutos"] if info else None,
                      "minutos de evaluación": m["minutos"]}, dtype=object)

---
### Preparación. El piso

El piso es la evaluación del modelo base sin entrenar con el prompt de `prompt_texto`, que no usa la plantilla de chat. Cada alumno se compara con esa evaluación.

La evaluación del piso no depende de ningún código tuyo. Por eso la cátedra la midió una vez, en una T4 de Colab, con las funciones de la preparación anterior. La función `cargar_medicion` de la celda que sigue descarga esa medición del repositorio de la materia. Si la descarga falla, la función evalúa el modelo en la sesión con `evaluar_o_cargar`, en unos 10 minutos de T4.

La celda crea además la tabla `comparacion`, con la fila del piso. Los ejercicios 2, 4 y 5 le agregan la fila de cada alumno. La preparación del Ejercicio 3 le agrega la de `Qwen3-0.6B`. Al final, la celda imprime un problema que el piso resuelve, con el texto que escribió el modelo.

| Nombre | Qué es |
|---|---|
| `cargar_medicion(nombre, cargar_modelo, armar)` | la función que descarga la medición de la cátedra del modelo `nombre`. Si la descarga falla, la función llama a `evaluar_o_cargar` con los mismos argumentos. En los dos casos, el resultado tiene las mismas claves que el de `evaluar_o_cargar` |
| `piso` | el resultado de `cargar_medicion` para el modelo base con prompt de texto |
| `comparacion` | un `DataFrame` con una fila por modelo y las columnas de `resumen` |

In [ ]:
# ─── Preparación. El piso: el modelo base, preguntado como texto ────────────
def cargar_medicion(nombre, cargar_modelo, armar, **config):
    """La evaluación que midió la cátedra en una T4. Si no se puede descargar, se evalúa en esta sesión."""
    archivo = f"lab4_{nombre}.jsonl"
    try:
        if not os.path.exists(archivo):
            urllib.request.urlretrieve(f"{DATOS}/{archivo}", archivo)
        with open(archivo, encoding="utf-8") as entrada:
            detalle = [json.loads(linea) for linea in entrada if linea.strip()][1:]   # la primera línea es la configuración
        if len(detalle) != len(prueba):
            raise ValueError(f"tiene {len(detalle)} problemas")
    except Exception as falla:
        if os.path.exists(archivo):
            os.remove(archivo)
        print(f"[{nombre}] la medición de la cátedra no está disponible ({falla}); se evalúa en esta sesión.")
        return evaluar_o_cargar(nombre, cargar_modelo, armar, **config)
    print(f"[{nombre}] medición de la cátedra, descargada del repositorio de la materia")
    return resultado(nombre, detalle)


piso = cargar_medicion("piso", lambda: cargar(BASE), prompt_texto, modelo=BASE)
comparacion = pd.DataFrame({"piso, modelo base con prompt de texto": resumen(piso)}).T
display(comparacion)
c2.mostrar(piso, "ok", 1, largo=600)

---
### Ejercicio 2. El entrenamiento con LoRA

**Objetivo.** Entrenar un alumno con la receta de la sección 3.3 de la Clase 1, sin cambios, y medirlo contra el piso.

Este ejercicio entrena el primer alumno con las primeras 100 trazas. Con 100 trazas, los 220 pasos de lotes efectivos de 8 trazas recorren cada traza unas 17 veces. El entrenamiento lleva del orden de 17 minutos en una T4, y la evaluación, del orden de 20.

La accuracy de este alumno puede quedar por debajo de la del piso. Los ejercicios siguientes analizan ese resultado.

**Enunciado:**

1. Entrená un alumno con `entrenar_alumno`, con el nombre `"alumno_100_lora"` y las primeras 100 trazas de `trazas`, sin el argumento `tokens_especiales`. Guardá el diccionario que devuelve la función en una variable llamada `info_100_lora`.
2. Evaluá el alumno con `evaluar_alumno(info_100_lora)` y guardá el resultado en una variable llamada `alumno_100_lora`.
3. Agregá a `comparacion` la `Series` que devuelve `resumen(alumno_100_lora, info_100_lora)`, como una fila nueva con la etiqueta `"100 trazas, LoRA"`. Mostrá `comparacion`.

In [ ]:
# Tu código aquí

---
### Preparación. El modelo post-entrenado

La celda que sigue carga la evaluación de `Qwen3-0.6B` con el prompt de `prompt_chat`, que es el mismo de los alumnos. El Ejercicio 3 compara sus respuestas con las del primer alumno. El Ejercicio 5 ubica a los tres alumnos entre el piso y este modelo.

La sección 2.1 de la Clase 2 midió `Qwen3-0.6B` con el razonamiento encendido sobre 100 problemas, con el muestreo que recomienda Qwen para ese modo. Esa medición no es comparable con las de este laboratorio, que usan decodificación voraz sobre los 250 problemas. Por eso la cátedra volvió a medir `Qwen3-0.6B` en las mismas condiciones que los alumnos. Así, la diferencia entre `Qwen3-0.6B` y un alumno depende solo de los pesos de cada modelo. Como la del piso, la celda descarga esa medición con `cargar_medicion`. Si la descarga falla, la celda evalúa el modelo en la sesión, en unos 20 minutos de T4.

| Nombre | Qué es |
|---|---|
| `post` | el resultado de `cargar_medicion` para `Qwen3-0.6B` con la plantilla de chat |

In [ ]:
# ─── Preparación. Qwen3-0.6B, en las mismas condiciones que los alumnos ─────
post = cargar_medicion("post", lambda: cargar(POST), prompt_chat, modelo=POST)
comparacion.loc["Qwen3-0.6B, post-entrenado"] = resumen(post)
comparacion

---
### Ejercicio 3. Los tokens especiales

**Objetivo.** Medir cómo terminan las respuestas del alumno del Ejercicio 2 y comparar las filas de los cuatro tokens especiales en la tabla de *embeddings* del modelo base y en la del modelo post-entrenado.

Cada resultado de `evaluar_o_cargar` guarda en la clave `"detalle"` un diccionario por problema, con las claves de `c2.evaluar`. Este ejercicio usa tres de esas claves. Cada una es una lista con un elemento por respuesta, y cada problema tiene una sola respuesta.

- `"textos"` tiene el texto generado. El texto conserva `<think>` y `</think>`, pero no `<|im_end|>`, que es un token de fin.
- `"tokens"` tiene la cantidad de tokens generados.
- `"forzado"` indica si `c2.evaluar` cerró el pensamiento. Cuando una respuesta consume el presupuesto con el pensamiento abierto, `c2.evaluar` agrega el texto de cierre de la sección 2.2 de la Clase 2, que termina en `</think>`, y le da 256 tokens más para responder.

La tabla de *embeddings* de entrada tiene una fila por token del vocabulario. El método `get_input_embeddings()` del modelo devuelve la capa de esa tabla. El atributo `weight` de esa capa es la tabla, un tensor de forma `(V, 1024)`, donde `V` es el tamaño del vocabulario. La fila del token de índice `i` es `weight[i]`. En Qwen3, esa tabla está atada a la capa de salida, como la del Laboratorio 2b. Que estén atadas significa que la capa de salida usa la misma matriz. El logit de cada token es entonces el producto escalar entre el vector de salida del último bloque y la fila de ese token. El método `get_output_embeddings()` devuelve la capa de salida.

Dos tensores que comparten la memoria tienen el mismo `data_ptr()`, la dirección de su primer elemento, como en el Ejercicio 2 del Laboratorio 1a.

**Enunciado:**

1. Escribí la función `cierres(m)`. La función recibe un resultado de `evaluar_o_cargar` y devuelve una `Series` con dos porcentajes, calculados sobre las respuestas de `m["detalle"]`.
   - `"cierran el pensamiento %"` es el porcentaje de respuestas cuyo texto contiene `</think>` y cuyo valor de `"forzado"` es `False`.
   - `"consumen el presupuesto %"` es el porcentaje de respuestas cuya cantidad de tokens es mayor o igual que `PRESUPUESTO`.
2. Construí un `DataFrame` llamado `tabla_cierres`, con la fila de `cierres(alumno_100_lora)` y la de `cierres(post)`, con las etiquetas de `comparacion`. Mostrá `tabla_cierres`.
3. Cargá el modelo base en la CPU, con los pesos en `float32`, con `cargar(BASE, torch.float32, "cpu")`.
   - Imprimí si la tabla de *embeddings* y la capa de salida comparten la memoria, comparando el `data_ptr()` del atributo `weight` de las dos capas.
   - Calculá el coseno entre cada par de filas de los cuatro tokens especiales, con los índices de `IDS_ESPECIALES`. El cálculo es el de la similitud coseno de la sección 6.2 de la Unidad 1. Guardá el resultado en un `DataFrame` de 4 × 4 llamado `cosenos_base`, con los textos de `ESPECIALES` como índice y como columnas.
   - Borrá el modelo con `del` y llamá a `c2.liberar()`, para liberar la memoria antes de cargar el siguiente.

   Repetí los tres pasos con `POST` y guardá su tabla en un `DataFrame` llamado `cosenos_post`. Mostrá las dos tablas con tres decimales.

In [ ]:
# Tu código aquí

**Pregunta de análisis:**

Con `tabla_cierres`, `cosenos_base` y `cosenos_post` a la vista, explicá por qué el alumno del Ejercicio 2 no escribe `</think>` ni `<|im_end|>` como lo hace `Qwen3-0.6B`, aunque el objetivo de cada traza los contiene.

*(Escribí tu respuesta acá)*

---
### Ejercicio 4. El entrenamiento de las filas de los tokens especiales

**Objetivo.** Entrenar un alumno con LoRA y con las filas de los cuatro tokens especiales, y comparar el resultado con el del Ejercicio 2.

PEFT (*parameter-efficient fine-tuning*, ajuste eficiente de parámetros) es la biblioteca de Hugging Face que implementa LoRA. La clase la usa sin presentarla. Su clase `LoraConfig` tiene el argumento `trainable_token_indices`, que no aparece en la teoría.

El argumento recibe una lista de índices de tokens. Con ese argumento, el entrenamiento ajusta las filas de esos tokens en la tabla de *embeddings*, además de las matrices de LoRA, y deja fijas las demás filas de la tabla. `entrenar_alumno` pasa su argumento `tokens_especiales` a ese argumento de `LoraConfig`.

```python
LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, task_type="CAUSAL_LM", target_modules=PROYECCIONES,
           trainable_token_indices=tokens_especiales)
```

Este alumno difiere del alumno del Ejercicio 2 en un solo factor, el entrenamiento de las cuatro filas. Las trazas, los pasos y la receta son los mismos. El entrenamiento y la evaluación llevan el mismo tiempo que en el Ejercicio 2.

**Enunciado:**

1. Entrená un alumno con `entrenar_alumno`, con el nombre `"alumno_100_filas"`, las primeras 100 trazas de `trazas` y `tokens_especiales=IDS_ESPECIALES`. Guardá el diccionario que devuelve la función en una variable llamada `info_100_filas`.
2. Evaluá el alumno con `evaluar_alumno` y guardá el resultado en una variable llamada `alumno_100_filas`. Agregá a `comparacion` la fila de `resumen(alumno_100_filas, info_100_filas)`, con la etiqueta `"100 trazas, LoRA y filas"`, y mostrá `comparacion`.
3. Agregá a `tabla_cierres` la fila de `cierres(alumno_100_filas)`, con la misma etiqueta, y mostrá `tabla_cierres`.

> **Nota. El umbral de diagnóstico.** Si la accuracy de este alumno queda por debajo de 46 %, el entrenamiento tiene un error. Revisá que tu función `objetivo` del Ejercicio 1 escriba `<think>`, `</think>` y `<|im_end|>` exactamente, y que la llamada del punto 1 pase `tokens_especiales`.

In [ ]:
# Tu código aquí

**Pregunta de análisis:**

Según la columna `"entrenables (miles)"` de `comparacion`, `trainable_token_indices` agrega unos pocos miles de parámetros a los diez millones de LoRA. Explicá por qué esos parámetros cambian cómo terminan las respuestas, y por qué entrenar más pasos con LoRA sola no lo cambiaría, a partir de qué matrices del modelo ajusta cada uno.

*(Escribí tu respuesta acá)*

---
### Ejercicio 5. La cantidad de trazas

**Objetivo.** Entrenar un alumno con las 491 trazas y la misma cantidad de pasos, y ubicar a los tres alumnos entre el piso y `Qwen3-0.6B`.

Con 491 trazas, los 220 pasos recorren cada traza unas 3,6 veces, contra unas 17 veces con 100 trazas. Este alumno difiere del alumno del Ejercicio 4 en un solo factor, la cantidad de trazas distintas. El entrenamiento y la evaluación llevan del orden de 20 minutos cada uno en una T4.

**Enunciado:**

1. Entrená un alumno con `entrenar_alumno`, con el nombre `"alumno_491_filas"`, todas las trazas de `trazas` y `tokens_especiales=IDS_ESPECIALES`. Guardá el diccionario que devuelve la función en una variable llamada `info_491_filas`. Evaluá el alumno con `evaluar_alumno` y guardá el resultado en una variable llamada `alumno_491_filas`. Agregá a `comparacion` la fila de `resumen(alumno_491_filas, info_491_filas)`, con la etiqueta `"491 trazas, LoRA y filas"`.
2. Agregá a `comparacion` una columna llamada `"avance %"`. El avance de un modelo es el porcentaje de la diferencia de accuracy entre el piso y `Qwen3-0.6B` que ese modelo agrega sobre el piso. Para un modelo con accuracy $a$, el avance es $100\,(a - a_{\text{piso}}) / (a_{\text{post}} - a_{\text{piso}})$, donde $a_{\text{piso}}$ es la accuracy de `piso` y $a_{\text{post}}$ es la de `post`. Mostrá `comparacion` con la columna redondeada a un decimal.
3. Contá con `c2.comparar` los problemas que `alumno_491_filas` resuelve y `alumno_100_filas` no, y los problemas que `alumno_100_filas` resuelve y `alumno_491_filas` no. Imprimí los dos números.

In [ ]:
# Tu código aquí

**Pregunta de análisis:**

Dos entrenamientos con la misma configuración y distinta semilla difieren en unos 2 puntos de accuracy sobre los 250 problemas. Con esa escala de ruido, ¿tus resultados permiten afirmar que pasar de 100 a 491 trazas, con la misma cantidad de pasos, mejora al alumno? Fundamentá la respuesta con la diferencia de accuracy, con el conteo del punto 3 y con la columna `"avance %"`.

*(Escribí tu respuesta acá)*

---
## Antes de entregar

Revisá esta lista:

- [ ] Reinicié el entorno y ejecuté **todas** las celdas de arriba a abajo sin errores (**Entorno de ejecución > Reiniciar y ejecutar todo**). Los entrenamientos y las evaluaciones guardados en Drive se cargan y no se repiten.
- [ ] En la tabla `etiquetas` del Ejercicio 1, `<think>` y `<|im_end|>` aparecen como un token cada uno.
- [ ] `comparacion` tiene cinco filas: el piso, `Qwen3-0.6B` y los tres alumnos.
- [ ] La accuracy del alumno del Ejercicio 4 es de 46 % o más.
- [ ] Respondí las preguntas de análisis de los ejercicios 1, 3, 4 y 5.
- [ ] No modifiqué ninguna celda fuera de las de actividad.

---
## Cierre

Este laboratorio entrenó un razonador con el procedimiento que la sección 3.4 de la Clase 2 describe para los modelos destilados de DeepSeek, a la escala de una T4. El entrenamiento ajustó el modelo base para escribir el pensamiento y la respuesta con el formato de las trazas de un maestro unas siete veces más grande. El entrenamiento usó LoRA y las filas de cuatro tokens especiales, sin refuerzo ni recompensas.

Dos prácticas del laboratorio se aplican a cualquier entrenamiento de un modelo preentrenado:

- verificar que los tokens que el objetivo de entrenamiento usa tengan filas entrenadas en el modelo de partida;
- comparar cada resultado con un piso medido en las mismas condiciones y con la escala del ruido entre corridas.